In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score, mean_absolute_error, mean_squared_error, 
    mean_absolute_percentage_error
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

# 출력 설정
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 100) 

# 시각화 설정
plt.rcParams['figure.figsize'] = (14, 8)
sns.set_style("whitegrid")

np.random.seed(42)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style("whitegrid")


In [3]:
# 데이터 로드 & 정제

all = pd.read_csv('../data/olist_preprocess_ver2_data.csv')
df_geo = pd.read_csv('../data/geolocation_clean.csv')
geo = df_geo.copy()
df= all.copy()
# print(df.shape[0], 'rows × ', df.shape[1], 'columns')
# 111495 rows ×  29 columns

# 날짜 칼럼 변환
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date'
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df = df[
    (df['order_delivered_customer_date'] != pd.Timestamp('2099-01-01')) &  #두 조건 동일행 의미
    (df['order_status'] == 'delivered')
]
# removed rows: 1881
# remaining rows: 10961

In [4]:
# 배송 지연 계산 (핵심 타겟)
# 배송 지연 일수 계산
df['delivery_delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
# df['is_delayed'] = (df['delivery_delay_days'] > 0).astype(int)

In [5]:
#극단, 이상치 확인
Q1 = df['delivery_delay_days'].quantile(0.25)
Q3 = df['delivery_delay_days'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
lower_outlier_count = (df['delivery_delay_days'] < lower_bound).sum()

upper_bound = Q3 + 1.5 * IQR
upper_outlier_count = (df['delivery_delay_days'] > upper_bound).sum()

# print((df['delivery_delay_days'] < lower_bound).sum()) 
# print((df['delivery_delay_days'] > upper_bound).sum())

# Lower: 2123 #전체의 약 2%
# Upper: 2680

In [6]:
#상한선 99.9 결정
# q999 = df['delivery_delay_days'].quantile(0.999) 
#len(df) #108790 rows
#df_cut = df[df['delivery_delay_days'] <= q999]
# len(df_cut)  #108681
# 289행 감소 #0.2%

In [6]:
# 상한선, 하한선 도입
# 분포 균형을 위해서 log 도입(0값으로 인해서 log1p)
q999 = df['delivery_delay_days'].quantile(0.999) 
df = df[(df['delivery_delay_days'] <= q999) & 
            (df['delivery_delay_days'] >= lower_bound)]
df['delay_log'] = np.log1p(df['delivery_delay_days'])

In [7]:
len(df)

107366

In [8]:
# 지리적 정보 병합 및 거리 계산
# 하버사인 공식으로 두 점 사이의 거리 계산
def calculate_distance(lat1, lng1, lat2, lng2):
    R = 6371  # 지구의 반지름 (km)
    
    lat1_rad = np.radians(lat1)
    lng1_rad = np.radians(lng1)
    lat2_rad = np.radians(lat2)
    lng2_rad = np.radians(lng2)
    
    dlat = lat2_rad - lat1_rad
    dlng = lng2_rad - lng1_rad
    
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlng/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c

# 고객 위치 좌표 병합
geo_customer = geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean'
}).reset_index()

geo_customer.columns = ['customer_zip_code_prefix', 'customer_lat', 'customer_lng']
df_merged = df.merge(geo_customer, on='customer_zip_code_prefix', how='left')

# 판매자 위치 좌표 병합
geo_seller = geo.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean'
}).reset_index()

geo_seller.columns = ['seller_zip_code_prefix', 'seller_lat', 'seller_lng']
df_merged = df_merged.merge(geo_seller, on='seller_zip_code_prefix', how='left')


In [9]:
# 각 주문에 대해 고객과 판매자 사이의 거리 계산
distances = []
for index, row in df_merged.iterrows():
    if pd.notna(row['seller_lat']) and pd.notna(row['customer_lat']):
        dist = calculate_distance(
            row['seller_lat'], row['seller_lng'],
            row['customer_lat'], row['customer_lng']
        )
        distances.append(dist)
    else:
        distances.append(None)

df_merged['actual_distance_km'] = distances

# 거리 정보가 없는 행 제거
df_merged = df_merged.dropna(subset=['actual_distance_km'])

print(f"거리 계산 후: {df_merged.shape[0]} rows")

거리 계산 후: 105087 rows


In [ ]:
# 파생 특성 생성

# 상품 부피 계산
df_merged['product_volume'] = (
    df_merged['product_length_cm'] * 
    df_merged['product_height_cm'] * 
    df_merged['product_width_cm']
)

# 주문 월과 요일 추출
df_merged['order_month'] = df_merged['order_purchase_timestamp'].dt.month
df_merged['order_dayofweek'] = df_merged['order_purchase_timestamp'].dt.dayofweek

# 시즌 결정 함수
def get_season(month):
    if month in [12, 1, 2]:
        return 'Summer'
    elif month in [3, 4, 5]:
        return 'Autumn'
    elif month in [6, 7, 8]:
        return 'Winter'
    else:
        return 'Spring'

# 각 주문의 시즌 결정
seasons = []
for month in df_merged['order_month']:
    season = get_season(month)
    seasons.append(season)

df_merged['season'] = seasons

# 주말 여부 결정 (금요일=4, 토요일=5, 월요일=0이므로 5이상이면 주말)
weekends = []
for day_of_week in df_merged['order_dayofweek']:
    if day_of_week >= 5:
        weekends.append(1)
    else:
        weekends.append(0)

df_merged['weekend'] = weekends

print(f"파생 특성 생성 완료: {df_merged.shape[1]} columns")

파생 특성 생성 완료: 41 columns


In [11]:
# 모델 학습용 데이터 준비
# 모든 배송 지연 데이터 포함 (음수, 0, 양수)
df_model = df_merged.copy()
print(f"최종 데이터: {df_model.shape[0]:,} rows")

최종 데이터: 105,087 rows


In [12]:
# 특성 선택 및 인코딩
# 사용할 특성 목록
features_list = [
    'price', 'freight_value', 'product_weight_g', 'product_volume',
    'product_category_name_english', 'actual_distance_km', 'order_month',
    'order_dayofweek', 'weekend', 'season'
]

# 결측치 제거
df_features = df_model[features_list + ['delivery_delay_days']].dropna()

print(f"선택 특성 (결측치 제거): {df_features.shape[0]:,} rows, {len(features_list)} features")

# 카테고리 변수 인코딩
label_encoder_season = LabelEncoder()
label_encoder_category = LabelEncoder()

df_features['season_encoded'] = label_encoder_season.fit_transform(df_features['season'])
df_features['category_encoded'] = label_encoder_category.fit_transform(df_features['product_category_name_english'])

선택 특성 (결측치 제거): 103,589 rows, 10 features


In [13]:
# 수치형 특성 목록
numeric_features = [
    'price', 'freight_value', 'product_weight_g', 'product_volume',
    'actual_distance_km', 'order_month', 'order_dayofweek', 'weekend',
    'season_encoded', 'category_encoded'
]

print(f"인코딩 완료: {len(numeric_features)} numeric features")

인코딩 완료: 10 numeric features


In [14]:
# 다중공산성 검사 (VIF)

X_vif = df_features[numeric_features].copy()

vif_list = []
for i in range(X_vif.shape[1]):
    vif = variance_inflation_factor(X_vif.values, i)
    vif_list.append({
        'Feature': numeric_features[i],
        'VIF': vif
    })

vif_result = pd.DataFrame(vif_list)
vif_result = vif_result.sort_values('VIF', ascending=False).reset_index(drop=True)

print("\nVIF 분석 결과:")
print(vif_result.to_string(index=False))


VIF 분석 결과:
           Feature      VIF
     freight_value 6.169692
   order_dayofweek 5.916942
    product_volume 4.343350
  product_weight_g 4.310286
       order_month 3.840320
  category_encoded 3.099230
           weekend 2.969132
actual_distance_km 2.689933
    season_encoded 2.542640
             price 1.759461


In [19]:
df_features[numeric_features]

,price,freight_value,product_weight_g,product_volume,actual_distance_km,order_month,order_dayofweek,weekend,season_encoded,category_encoded
0,29.99,8.72,500.0,1976.0,18.576110,10,0,0,1,49
2,159.90,19.22,420.0,9576.0,514.410666,8,2,0,3,5
3,45.00,27.20,450.0,6000.0,1822.226336,11,5,1,1,60
4,19.90,8.72,250.0,11475.0,29.676625,2,1,0,2,66
5,147.90,27.36,7150.0,42250.0,411.394362,7,6,1,3,5
...,...,...,...,...,...,...,...,...,...,...
107361,174.90,20.10,4950.0,16000.0,474.120037,2,1,0,2,6
107362,205.99,65.02,13300.0,63360.0,967.847297,8,6,1,3,45
107363,179.99,40.59,6550.0,8000.0,370.404482,1,0,0,2,15
107364,179.99,40.59,6550.0,8000.0,370.404482,1,0,0,2,15


In [ ]:
# 데이터 불균형 처리 - 샘플 가중치
# 배송 지연 정도에 따라 가중치 부여
# sample_weights = np.abs(df_features['delivery_delay_days']) + 1
# 결과 
# 가중치 통계:
# 최소값: 1.0000
# 최대값: 54.0000
# 평균값: 13.7825

#로그 변환 선택!
sample_weights = np.log1p(np.abs(df_features['delivery_delay_days']))

# print(f"최소값: {sample_weights.min():.4f}")
# print(f"최대값: {sample_weights.max():.4f}")
# print(f"평균값: {sample_weights.mean():.4f}")
# 결과
# 가중치 통계:
# 최소값: 0.0000
# 최대값: 3.9890
# 평균값: 2.4622

가중치 통계:
최소값: 0.0000
최대값: 3.9890
평균값: 2.4622


In [18]:
# 데이터 분할 (Train/Test)

X = df_features[numeric_features]
y = df_features['delivery_delay_days']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 가중치도 같은 방식으로 분할
train_indices = X_train.index
test_indices = X_test.index
train_weights = sample_weights[train_indices].values
test_weights = sample_weights[test_indices].values

print(f"Train: {len(X_train):,} rows ({len(X_train)/len(X)*100:.1f}%)")
print(f"Test:  {len(X_test):,} rows ({len(X_test)/len(X)*100:.1f}%)")


Train: 82,871 rows (80.0%)
Test:  20,718 rows (20.0%)


In [ ]:
# 선형 회귀 (Linear Regression) + GridSearchCV

# Pipeline 생성
pipeline_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

# 하이퍼파라미터 
param_grid_lr = {
    'model__fit_intercept': [True]
}

# GridSearchCV
grid_lr = GridSearchCV(
    pipeline_lr,
    param_grid_lr,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

# 모델 학습
grid_lr.fit(X_train, y_train, model__sample_weight=train_weights)
print(f"최고 CV 점수 (R²), {grid_lr.best_score_:.4f}")

# CV 결과 
cv_scores = grid_lr.cv_results_['split0_test_score']
for i in range(5):
    fold_score = grid_lr.cv_results_[f'split{i}_test_score'][0]
    print(f"   Fold {i+1}: {fold_score:.4f}")
print(f"평균, {grid_lr.best_score_:.4f}")
print(f"표준편차, {grid_lr.cv_results_['std_test_score'][0]:.4f}")

# Train 예측
y_pred_train_lr = grid_lr.predict(X_train)

# 평가 지표 계산 (Train)
train_r2_lr = r2_score(y_train, y_pred_train_lr, sample_weight=train_weights)
train_mae_lr = mean_absolute_error(y_train, y_pred_train_lr, sample_weight=train_weights)
train_rmse_lr = np.sqrt(mean_squared_error(y_train, y_pred_train_lr, sample_weight=train_weights))

print(f"R² Score, {train_r2_lr}")
print(f"MAE (일), {train_mae_lr}")
print(f"RMSE (일), {train_rmse_lr:>15.4f}")

# 특성 중요도 (선형 회귀의 계수 절댓값)
model_lr = grid_lr.best_estimator_.named_steps['model']
importance_data_lr = []
for feature, coef in zip(numeric_features, model_lr.coef_):
    importance_data_lr.append({
        'Feature': feature,
        'Importance': np.abs(coef)
    })

feature_importance_lr = pd.DataFrame(importance_data_lr)
feature_importance_lr = feature_importance_lr.sort_values('Importance', ascending=False)

for i, (idx, row) in enumerate(feature_importance_lr.head(5).iterrows(), 1):
    print(f"   {i}. {row['Feature']} | {row['Importance']:.6f}")



# 최고 CV 점수 (R²): -0.0137
# Cross-Validation 상세 결과
#    Fold 1: -0.0174
#    Fold 2: -0.0230
#    Fold 3: -0.0119
#    Fold 4: -0.0108
#    Fold 5: -0.0055
# 평균: -0.0137
# 표준편차: 0.0060

# 평가 지표 (Train 데이터):
# R² Score                                0.0170
# MAE (일)                                 6.0305
# RMSE (일)                                8.8320

# 특성 중요도 (상위 5개):
#    1. season_encoded                  0.823161
#    2. actual_distance_km              0.574068
#    3. order_month                     0.427010
#    4. freight_value                   0.327972
#    5. product_weight_g                0.218761

최고 CV 점수 (R²), -0.0137
   Fold 1: -0.0174
   Fold 2: -0.0230
   Fold 3: -0.0119
   Fold 4: -0.0108
   Fold 5: -0.0055
평균, -0.0137
표준편차, 0.0060
R² Score, 0.017046530143389083
MAE (일), 6.030526470536627
RMSE (일),          8.8320
   1. season_encoded | 0.823161
   2. actual_distance_km | 0.574068
   3. order_month | 0.427010
   4. freight_value | 0.327972
   5. product_weight_g | 0.218761


In [ ]:
# 최종 평가: Test 데이터로 성능 확인 

# Test 데이터 예측
y_pred_test_lr = grid_lr.predict(X_test)

# 평가 지표 계산 (Test)
test_r2_lr = r2_score(y_test, y_pred_test_lr, sample_weight=test_weights)
test_mae_lr = mean_absolute_error(y_test, y_pred_test_lr, sample_weight=test_weights)
test_rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_test_lr, sample_weight=test_weights))

print(f"R² Score, {train_r2_lr:>15.4f}, {test_r2_lr:>15.4f}")
print(f"MAE (일), {train_mae_lr:>15.4f}, {test_mae_lr:>15.4f}")
print(f"RMSE (일), {train_rmse_lr:>15.4f}, {test_rmse_lr:>15.4f}")


# 과적합 판단
r2_diff_lr = abs(train_r2_lr - test_r2_lr)
print(f"Train R² - Test R²: {r2_diff_lr:.4f}")

if r2_diff_lr < 0.05:
    print("정상 (과적합 없음)")
elif r2_diff_lr < 0.10:
    print("약간의 과적합")
else:
    print("과적합 주의")


# 평가 지표 (Train vs Test):
# ----------------------------------------------------------------------
# 지표                                       Train            Test
# ----------------------------------------------------------------------
# R² Score                                0.0170          0.0147
# MAE (일)                                 6.0305          6.0304
# RMSE (일)                                8.8320          8.7834
# ----------------------------------------------------------------------
# 과적합 분석:
# Train R² - Test R²: 0.0024
# 정상 (과적합 없음)

In [24]:
import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import cross_validate

In [ ]:
# 랜덤 포레스트 (Random Forest) + Optuna 최적화

# Optuna 목표 함수 정의 
def objective(trial):
# 파라미터 제안
    n_estimators = trial.suggest_int('n_estimators', 100, 250)
    max_depth = trial.suggest_int('max_depth', 10, 20)  
    min_samples_split = trial.suggest_int('min_samples_split', 5, 15)  
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 8)  
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2'])
    
    # Pipeline 생성
    pipeline_rf = Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    # Cross-validation으로 평가
    try:
        cv_results = cross_validate(
            pipeline_rf,
            X_train,
            y_train,
            cv=5,
            scoring='r2'
        )
        return cv_results['test_score'].mean()
    
    except Exception as e:
        return 0.0

# Pruner 설정
sampler = optuna.samplers.TPESampler(seed=42)
pruner = MedianPruner()

study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner
)

# 모델 학습
study.optimize(
    objective,
    n_trials=50,
    show_progress_bar=True
)

print(f"최고 CV 점수 (R²): {study.best_value:.4f}")
print(f"\n최적 하이퍼파라미터")
for param, value in study.best_params.items():
    print(f"- {param}: {value}")

# 최고의 파라미터로 최종 모델 생성
best_params = study.best_params

final_pipeline_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(
        n_estimators=best_params['n_estimators'],
        max_depth=best_params['max_depth'],
        min_samples_split=best_params['min_samples_split'],
        min_samples_leaf=best_params['min_samples_leaf'],
        max_features=best_params['max_features'],
        random_state=42,
        n_jobs=-1
    ))
])

# 최종 모델 학습 (Train으로!)
final_pipeline_rf.fit(X_train, y_train, model__sample_weight=train_weights)


# Train 성능 확인 (참고용)
y_pred_train_rf = final_pipeline_rf.predict(X_train)

train_r2_rf = r2_score(y_train, y_pred_train_rf, sample_weight=train_weights)
train_mae_rf = mean_absolute_error(y_train, y_pred_train_rf, sample_weight=train_weights)
train_rmse_rf = np.sqrt(mean_squared_error(y_train, y_pred_train_rf, sample_weight=train_weights))

print(f"R² Score, {train_r2_rf:>15.4f}")
print(f"MAE (일), {train_mae_rf:>15.4f}")
print(f"RMSE (일), {train_rmse_rf:>15.4f}")

# 특성 중요도
model_rf = final_pipeline_rf.named_steps['model']
importance_data_rf = []
for feature, importance in zip(numeric_features, model_rf.feature_importances_):
    importance_data_rf.append({
        'Feature': feature,
        'Importance': importance
    })

feature_importance_rf = pd.DataFrame(importance_data_rf)
feature_importance_rf = feature_importance_rf.sort_values('Importance', ascending=False)

for i, (idx, row) in enumerate(feature_importance_rf.head(5).iterrows(), 1):
    print(f"{i}. {row['Feature']} | {row['Importance']:.6f}")


#    최고 CV 점수 (R²): 0.2194

#    최적 하이퍼파라미터:
#       - n_estimators: 233
#       - max_depth: 20
#       - min_samples_split: 5
#       - min_samples_leaf: 2
#       - max_features: sqrt

# ================================================================================
# Train 데이터에서의 성능 
# 평가 지표:
# ----------------------------------------------------------------------
# 지표                                       Train
# ----------------------------------------------------------------------
# R² Score                                0.6200
# MAE (일)                                 3.6739
# RMSE (일)                                5.4914
# ----------------------------------------------------------------------

# 특성 중요도 (상위 5개):
#    1. actual_distance_km             | 0.190091
#    2. freight_value                  | 0.156970
#    3. price                          | 0.135375
#    4. product_volume                 | 0.132501
#    5. product_weight_g               | 0.117819


In [ ]:
# 최종 평가: Test 데이터

# Test 데이터에 대한 예측
y_pred_test_rf = final_pipeline_rf.predict(X_test)

# 평가 지표 계산 (Test)
test_r2_rf = r2_score(y_test, y_pred_test_rf, sample_weight=test_weights)
test_mae_rf = mean_absolute_error(y_test, y_pred_test_rf, sample_weight=test_weights)
test_rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_test_rf, sample_weight=test_weights))

print(f"R² Score, {train_r2_rf:>15.4f} {test_r2_rf:>15.4f}")
print(f"MAE (일), {train_mae_rf:>15.4f} {test_mae_rf:>15.4f}")
print(f"RMSE (일), {train_rmse_rf:>15.4f} {test_rmse_rf:>15.4f}")

# 과적합 판단
r2_diff_rf = abs(train_r2_rf - test_r2_rf)
print(f"Train R² - Test R²: {r2_diff_rf:.4f}")
if r2_diff_rf < 0.05:
    print("정상 (과적합 없음)")
elif r2_diff_rf < 0.10:
    print("약간의 과적합")
else:
    print("과적합 주의")

# ================================================================================
# 최종 평가: Test 데이터
# ================================================================================

# 평가 지표 (Train vs Test):
# ----------------------------------------------------------------------
# 지표                                       Train            Test
# ----------------------------------------------------------------------
# R² Score                                0.6200          0.2369
# MAE (일)                                 3.6739          5.2222
# RMSE (일)                                5.4914          7.7297
# ----------------------------------------------------------------------
# 과적합 분석:
#    Train R² - Test R²: 0.3831
# 과적합 주의


최종 평가: Test 데이터

평가 지표 (Train vs Test):
----------------------------------------------------------------------
지표                                       Train            Test
----------------------------------------------------------------------
R² Score                                0.6200          0.2369
MAE (일)                                 3.6739          5.2222
RMSE (일)                                5.4914          7.7297
----------------------------------------------------------------------
과적합 분석:
   Train R² - Test R²: 0.3831
과적합 주의


In [ ]:
# 그래디언트 부스팅 (Gradient Boosting) + Optuna 최적화

import optuna
from optuna.pruners import MedianPruner
from sklearn.model_selection import cross_validate

# Optuna 목표 함수 정의 
def objective(trial):
    # 파라미터
    n_estimators = trial.suggest_int('n_estimators', 100, 200)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1)
    max_depth = trial.suggest_int('max_depth', 3, 6) 
    min_samples_split = trial.suggest_int('min_samples_split', 5, 15)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 2, 6)
    subsample = trial.suggest_float('subsample', 0.6, 1.0)  
    
    # Pipeline 
    pipeline_gb = Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingRegressor(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            subsample=subsample,
            random_state=42
        ))
    ])
    
    # Cross-validation 평가 
    try:
        cv_results = cross_validate(
            pipeline_gb,
            X_train,
            y_train,
            cv=5,
            scoring='r2'
        )
        
        return cv_results['test_score'].mean()
    except Exception as e:
        return 0.0

# Optuna Study 생성
# Pruner 설정 
sampler = optuna.samplers.TPESampler(seed=42)
pruner = MedianPruner()

study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner
)

# 모델 학습
study.optimize(
    objective,
    n_trials=50,
    show_progress_bar=True
)


print(f"최고 CV 점수 (R²): {study.best_value:.4f}")
for param, value in study.best_params.items():
    print(f"- {param}: {value}")

# CV 결과 
for i in range(5):
    fold_score = study.trials[0].intermediate_values.get(i, None)
    # Optuna의 개별 fold 접근은 복잡하므로, best_value로 평가
print(f"평균 CV R²: {study.best_value:.4f}")

# 최고의 파라미터로 최종 모델 생성
best_params = study.best_params
final_pipeline_gb = Pipeline([
    ('scaler', StandardScaler()),
    ('model', GradientBoostingRegressor(
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        max_depth=best_params['max_depth'],
        min_samples_split=best_params['min_samples_split'],
        min_samples_leaf=best_params['min_samples_leaf'],
        subsample=best_params['subsample'],
        random_state=42
    ))
])

# 최종 모델 학습
final_pipeline_gb.fit(X_train, y_train, model__sample_weight=train_weights)

# Train 성능 확인 (참고용)
# ============================================================================
y_pred_train_gb = final_pipeline_gb.predict(X_train)

train_r2_gb = r2_score(y_train, y_pred_train_gb, sample_weight=train_weights)
train_mae_gb = mean_absolute_error(y_train, y_pred_train_gb, sample_weight=train_weights)
train_rmse_gb = np.sqrt(mean_squared_error(y_train, y_pred_train_gb, sample_weight=train_weights))

print(f"R² Score, {train_r2_gb:>15.4f}")
print(f"MAE (일), {train_mae_gb:>15.4f}")
print(f"RMSE (일), {train_rmse_gb:>15.4f}")

# 특성 중요도
model_gb = final_pipeline_gb.named_steps['model']
importance_data_gb = []
for feature, importance in zip(numeric_features, model_gb.feature_importances_):
    importance_data_gb.append({
        'Feature': feature,
        'Importance': importance
    })

feature_importance_gb = pd.DataFrame(importance_data_gb)
feature_importance_gb = feature_importance_gb.sort_values('Importance', ascending=False)

for i, (idx, row) in enumerate(feature_importance_gb.head(5).iterrows(), 1):
    print(f"{i}. {row['Feature']} | {row['Importance']:.6f}")

print(f"Train 데이터에 sample_weight 적용 {train_weights is not None}")


#    최고 CV 점수 (R²): 0.1615

#    최적 하이퍼파라미터:
#       - n_estimators: 193
#       - learning_rate: 0.09199416457849235
#       - max_depth: 6
#       - min_samples_split: 11
#       - min_samples_leaf: 6
#       - subsample: 0.7213692618241557

# Cross-Validation 상세 결과:
#    평균 CV R²: 0.1615

# ================================================================================
# Train 데이터에서의 성능 (참고용 - 과적합 확인용)

# 평가 지표:
# ----------------------------------------------------------------------
# 지표                                       Train
# ----------------------------------------------------------------------
# R² Score                                0.2696
# MAE (일)                                 5.1913
# RMSE (일)                                7.6132
# ----------------------------------------------------------------------

# 특성 중요도 (상위 5개):
#    1. order_month                    | 0.212004
#    2. actual_distance_km             | 0.201835
#    3. freight_value                  | 0.154000
#    4. product_volume                 | 0.100887
#    5. price                          | 0.096208

# ================================================================================
# 가중치 적용 확인:
#    Train 데이터에 sample_weight 적용됨: True

In [ ]:
# 최종 평가: Test 데이터
# ============================================================================

# Test 데이터에 대한 예측
y_pred_test_gb = final_pipeline_gb.predict(X_test)

# 평가 지표 계산 (Test)
test_r2_gb = r2_score(y_test, y_pred_test_gb, sample_weight=test_weights)
test_mae_gb = mean_absolute_error(y_test, y_pred_test_gb, sample_weight=test_weights)
test_rmse_gb = np.sqrt(mean_squared_error(y_test, y_pred_test_gb, sample_weight=test_weights))

print(f"R² Score, {train_r2_gb:>15.4f} {test_r2_gb:>15.4f}")
print(f"MAE (일), {train_mae_gb:>15.4f} {test_mae_gb:>15.4f}")
print(f"RMSE (일), {train_rmse_gb:>15.4f} {test_rmse_gb:>15.4f}")

# 과적합 판단
r2_diff_gb = abs(train_r2_gb - test_r2_gb)
print(f"\n과적합 분석:")
print(f"Train R² - Test R²: {r2_diff_gb:.4f}")

if r2_diff_gb < 0.05:
    print(" 정상 (과적합 없음)")
elif r2_diff_gb < 0.10:
    print("약간의 과적합")
else:
    print("과적합 주의")



# ================================================================================
# 최종 평가: Test 데이터

# 평가 지표 (Train vs Test):
# ----------------------------------------------------------------------
# 지표                                       Train            Test
# ----------------------------------------------------------------------
# R² Score                                0.2696          0.1738
# MAE (일)                                 5.1913          5.4949
# RMSE (일)                                7.6132          8.0431
# ----------------------------------------------------------------------

# 과적합 분석:
#    Train R² - Test R²: 0.0959
#    약간의 과적합
